# Notebook08 — fixed10-pair energy encoder replication
Definitions only. Main Notebook08 starts a detached worker with an immutable8-hour clock including calibration, exact full-state resume, actual evaluation integration,20trainingarms and40fixedprobes. Reuses unchanged Notebook07 encoder. No adaptive seed/update selection or stopping at significance. Artifacts: artifacts/notebook_08/<run_id>/. Full checkpoints: checkpoints/<run_id>/. Resume via unchanged Notebook08 Run All before its original deadline. Official test reserved.

In [ ]:
"""Notebook08 orchestration: one persisted eight-hour experimental clock."""
from pathlib import Path
import copy, hashlib, importlib.util, json, os, shlex, subprocess, sys, time
from datetime import datetime, timezone
import nbformat
from fmm_jepa.bootstrap import TaggedNotebookLoader
from fmm_jepa.training import atomic_json, digest_file, run_writer
MAIN_NOTEBOOK='notebooks/08_Energy_Encoder_Replication.ipynb'
TERMINAL={'COMPLETED','FAILED','INCOMPLETE'}

def _read(path):return json.loads(Path(path).read_text())
def _normalized(value):return json.loads(json.dumps(value,allow_nan=False))
def _hash(value):return hashlib.sha256(json.dumps(_normalized(value),sort_keys=True,separators=(',',':')).encode()).hexdigest()
def _export_hash(path):
    return hashlib.sha256('\n'.join(''.join(c['source']) for c in _read(path)['cells'] if c['cell_type']=='code' and 'ahr-export' in c.get('metadata',{}).get('tags',[])).encode()).hexdigest()
def _load(root,name):
    key='fmm08_'+name
    if key not in sys.modules:
        spec=importlib.util.spec_from_loader(key,TaggedNotebookLoader(Path(root)/'experiments/08_replication'/(name+'.ipynb')))
        module=importlib.util.module_from_spec(spec);sys.modules[key]=module
        try:spec.loader.exec_module(module)
        except BaseException:sys.modules.pop(key,None);raise
    return sys.modules[key]

def source_hashes(root):
    return {str(p.relative_to(root)):_export_hash(p) for folder in ('fmm_jepa','experiments/04_followup','experiments/05_scaling','experiments/07_energy','experiments/08_replication') for p in sorted((Path(root)/folder).glob('*.ipynb'))}

def consumer_hash(root):return _hash([''.join(c['source']) for c in _read(Path(root)/MAIN_NOTEBOOK)['cells'] if c['cell_type']=='code'])

def study_configuration(run_id='energy-replication-001'):
    if not run_id or any(not(c.isalnum() or c in '-_') for c in run_id):raise ValueError('Safe nonempty run ID required')
    return {'schema':'fmm-jepa-notebook08-study/v1','run_id':run_id,'hard_limit_seconds':28800,
        'device':'cuda:0','dtype':'float32','image_size':128,'effective_batch':128,'microbatch':128,
        'seeds':list(range(20261101,20261111)),'variants':['vit','energy'],'updates':350,
        'round_updates':50,'probe_epochs':20,'heldout_enabled':False,
        'energy':{'summary_dim':32,'steps':4,'near_strength':.25,'far_strength':.25,'step_size':.5,'coefficient_dim':32},
        'primary_analysis':{'endpoint':'Final development top1 accuracy, energy minus ViT, percentage points',
            'test':'paired two-sided Student t','alpha':.05,'confidence_level':.95,'n_pairs':10,'df':9,
            'planning_true_difference_pp':.5,'planning_power':.9,'pilot_sd_pp':.4200634343682461,
            'power_caveat':'Power assumes pilot variance transfers; two pilot pairs give an uncertain variance estimate.'},
        'secondary_analysis':'Own-initialization gains, adjusted gains, time, memory and representation diagnostics are exploratory.',
        'budget_policy':'Fixed10freshpairs x350updates; no reduction or extension.10percent timing allowance plus4000seconds evaluation and480seconds preflight/overhead. Deadline cutoff leaves INCOMPLETE.',
        'selection':'Architecture,350updates,epoch20probes,primarytest and10newseeds fixed before new quality outcomes. No stopping at significance.',
        'interpretation':'Replication over independent training seeds conditional on reused development membership. Original pilot excluded. Official test reserved; not a new-dataset confirmation.'}

def _status(artifact,stage,status,**details):
    result={'stage':stage,'status':status,'updated_unix':time.time(),**details}
    atomic_json(Path(artifact)/'study_status.json',result)
    print(stage+': '+status,flush=True)
    return result

def _validation(root):
    operator=Path(root)/'artifacts/notebook_07_validation/operator_validation.json'
    report=_read(operator);current=_export_hash(Path(root)/'experiments/07_energy/encoder.ipynb')
    if report.get('status')!='PASS' or report.get('encoder_export_sha256')!=current:
        raise ValueError('Unchanged Notebook07 energy operator must have matching PASS evidence')
    pipeline=Path(root)/'artifacts/notebook_08_validation/runner_evaluation_validation.json'
    validation=_read(pipeline)
    if validation.get('status')!='PASS':raise ValueError('Notebook08 runner/evaluation validation must pass before launch')
    for name in ('runner','evaluation'):
        if validation.get(name+'_export_sha256')!=_export_hash(Path(root)/'experiments/08_replication'/(name+'.ipynb')):
            raise ValueError('Pipeline validation does not match current '+name+' definitions')
    return {'operator':{'path':str(operator.relative_to(root)),'sha256':digest_file(operator),'encoder_export_sha256':current},
        'runner_evaluation':{'path':str(pipeline.relative_to(root)),'sha256':digest_file(pipeline)}}

def _check_binding(root,binding,config):
    reference='artifacts/notebook_07/energy-encoder-001/stages/learning/plan.json'
    if binding.get('reference_pilot_plan')!={'path':reference,'sha256':digest_file(Path(root)/reference)}:
        raise ValueError('The pilot configuration source changed')
    if binding['configuration']!=_normalized(config):raise ValueError('Registered run configuration changed')
    if binding['source_hashes']!=source_hashes(root):raise ValueError('Registered source definitions changed; use a new run ID')
    if binding['consumer_notebook_sha256']!=consumer_hash(root):raise ValueError('Registered main notebook code changed')
    if binding['operator_validation']!=_validation(root):raise ValueError('Bound validation evidence changed')

def _clock(artifact,binding):
    result=_read(Path(artifact)/'budget_clock.json')
    if result!=binding['budget_clock'] or result['max_seconds']!=28800:raise ValueError('Original eight-hour clock changed')
    return result

def _prepare(root,config):
    if _normalized(config)!=_normalized(study_configuration(config['run_id'])):
        raise ValueError('This replication has a fixed registered configuration; no seed/update/endpoint edits are permitted')
    root=Path(root).resolve();artifact=root/'artifacts/notebook_08'/config['run_id'];artifact.mkdir(parents=True,exist_ok=True)
    path=artifact/'study_binding.json'
    if path.exists():
        binding=_read(path);_check_binding(root,binding,config)
    else:
        if (artifact/'budget_clock.json').exists() or (artifact/'stages').exists():raise ValueError('Unbound existing experiment evidence; refusing restart')
        validation=_validation(root);now=time.time()
        clock={'started_unix':now,'deadline_unix':now+28800,'max_seconds':28800,'policy':'Original launch clock includes calibration, recovery verification, training and evaluation; never resets.'}
        binding={'schema':'notebook08-binding/v1','configuration':_normalized(config),'source_hashes':source_hashes(root),
                 'consumer_notebook_sha256':consumer_hash(root),'budget_clock':clock,'operator_validation':validation,
                 'reference_pilot_plan':{'path':'artifacts/notebook_07/energy-encoder-001/stages/learning/plan.json',
                    'sha256':digest_file(root/'artifacts/notebook_07/energy-encoder-001/stages/learning/plan.json')}}
        atomic_json(path,binding);atomic_json(artifact/'budget_clock.json',clock)
    return artifact,binding,_clock(artifact,binding)

def _choose_budget(config,calibration,deadline):
    if calibration['selected_microbatch']!=128 or calibration['selected_accumulation_steps']!=1:
        raise ValueError('Replication must retain pilot physical/effective batch128')
    if set(calibration['selected_model_step_seconds'])!={'vit','energy'}:raise ValueError('Missing calibration arm')
    per_pair=sum(calibration['selected_model_step_seconds'].values())
    if per_pair<=0:raise ValueError('Invalid calibration time')
    remaining=deadline-time.time();count=len(config['seeds']);updates=config['updates']
    reserve=4000;preflight=300;overhead=180
    training=1.10*count*updates*per_pair
    result={'status':'PASS','seeds':count,'updates':updates,'evaluation_reserve_seconds':reserve,
        'predicted_training_seconds':training,'predicted_total_remaining_seconds':training+reserve+preflight+overhead,
        'remaining_seconds':remaining,'calibration_round_seconds':per_pair,'fixed_schedule':True}
    if result['predicted_total_remaining_seconds']>remaining:
        raise TimeoutError('Fixed10-pair350-update replication does not fit original8-hour clock; no smaller subset or schedule will be substituted: '+str(result))
    return result

def _plan(root,study,binding,bundle,base,calibration,budget):
    preparation=_load(root,'preparation');arms=[];seeds=study['seeds'];updates=study['updates']
    for index,seed in enumerate(seeds):
        for variant in study['variants']:
            cfg=preparation.arm_config(base,variant)
            cfg['training'].update(updates=updates,microbatch=calibration['selected_microbatch'],accumulation_steps=calibration['selected_accumulation_steps'])
            cfg['seeds'].update(model_init=seed,data_order=311101+1000*index,augmentation=422202+1000*index,mask=533303+1000*index,probe=4404)
            cfg['model']['feature_seed']=seed+1000
            cfg['evaluation'].update(batch_size=min(64,calibration['selected_microbatch']),diagnostic_steps=[0,updates])
            cfg['budget']['evaluation_reserve_seconds']=budget['evaluation_reserve_seconds']
            arms.append({'id':'s'+str(seed)+'-'+variant,'kind':'vit' if variant=='vit' else 'semantic','variant':variant,'seed':seed,'condition':variant,'config':cfg})
    return _normalized({'schema':'notebook08-stage-plan/v1','run_id':study['run_id'],'stage_id':'learning','stage_role':'learning',
         'stage':{'id':'learning','image_size':128,'pretrain_pool':'selected'},'updates':updates,'milestones':[updates],
         'round_updates':study['round_updates'],'effective_batch':128,'seeds':seeds,'variants':study['variants'],'arms':arms,
         'train_count':len(bundle['train_ids']),'probe_train_count':len(bundle['probe_train_ids']),
         'source_hashes':binding['source_hashes'],'consumer_notebook_sha256':binding['consumer_notebook_sha256'],
         'resume_notebook':MAIN_NOTEBOOK,'resume_cell':'launch-energy-replication','evaluation_reserve_seconds':budget['evaluation_reserve_seconds'],
         'study_configuration_hash':_hash(study),'numerical_backend':'energy08-v1','heldout_status':'NOT_RUN',
         'primary_analysis':study['primary_analysis'],'secondary_analysis':study['secondary_analysis'],
         'excluded_pilot_seeds':[20261011,20261012],
         'selection':study['selection'],
         'failure_policy':'Stop first numerical failure; never skip samples.','budget_selection':budget})

def run_study(root,run_id):
    root=Path(root).resolve();artifact=root/'artifacts/notebook_08'/run_id
    binding=_read(artifact/'study_binding.json');study=binding['configuration'];_check_binding(root,binding,study)
    clock=_clock(artifact,binding);deadline=clock['deadline_unix']
    if (artifact/'study_status.json').exists() and _read(artifact/'study_status.json').get('status')=='COMPLETED':return _read(artifact/'study_status.json')
    if time.time()>=deadline:raise TimeoutError('Original eight-hour deadline expired')
    preparation,runner,evaluation=(_load(root,n) for n in ('preparation','runner','evaluation'))
    with run_writer(artifact):
        try:
            folder=artifact/'stages/learning';folder.mkdir(parents=True,exist_ok=True)
            (folder/'predictions').mkdir(exist_ok=True)
            base=preparation.base_config(root)
            base['model'].update({'energy_'+key:value for key,value in study['energy'].items()})
            _status(artifact,'verify_training_and_development_data','RUNNING')
            bundle=preparation.prepare_bundle(root,folder,base)
            _status(artifact,'actual_update_calibration','RUNNING')
            calibration=preparation.calibrate(root,folder,base,bundle,deadline)
            plan_path=folder/'plan.json'
            if plan_path.exists():
                plan=_read(plan_path)
                if plan['source_hashes']!=binding['source_hashes']:raise ValueError('Saved plan sources changed')
                if plan['study_configuration_hash']!=_hash(study):raise ValueError('Saved plan study changed')
            else:
                budget=_choose_budget(study,calibration,deadline)
                plan=_plan(root,study,binding,bundle,base,calibration,budget)
                preparation.immutable(plan_path,plan)
            _status(artifact,'exact_resume_preflight','RUNNING',updates=plan['updates'],seeds=plan['seeds'],variants=plan['variants'])
            runner.prepare_stage(root,folder,plan,bundle)
            preflight=runner.verify_stage_resume(root,folder,plan,bundle,deadline)
            atomic_json(folder/'preflight_result.json',preflight)
            if preflight.get('status')!='PASS':raise RuntimeError('Exact-resume preflight did not pass')
            _status(artifact,'evaluation_integration_preflight','RUNNING')
            scoring=evaluation.verify_evaluation_smoke(root,folder,plan,bundle,deadline)
            atomic_json(folder/'evaluation_preflight_result.json',scoring)
            if scoring.get('status')!='PASS':raise RuntimeError('Evaluation integration preflight failed')
            _check_binding(root,binding,study)
            _status(artifact,'paired_training','RUNNING',updates=plan['updates'],seeds=plan['seeds'],variants=plan['variants'])
            training=runner.train_stage(root,folder,plan,bundle,deadline)
            atomic_json(folder/'training_result.json',training)
            _status(artifact,'frozen_feature_evaluation','RUNNING',updates=plan['updates'],seeds=plan['seeds'])
            result=evaluation.evaluate_stage(root,folder,plan,bundle,training,deadline)
            atomic_json(artifact/'study_results.json',result)
            if result.get('status') not in ('PASS','COMPLETE','COMPLETED'):raise TimeoutError('Evaluation incomplete under original deadline')
            _check_binding(root,binding,study)
            return _status(artifact,'complete','COMPLETED',elapsed_seconds=time.time()-clock['started_unix'],updates=plan['updates'],seeds=plan['seeds'],variants=plan['variants'],heldout_status='NOT_RUN',results=str(folder/'stage_results.md'))
        except BaseException as error:
            _status(artifact,'interrupted','INCOMPLETE' if isinstance(error,TimeoutError) else 'FAILED',error=type(error).__name__+': '+str(error),cause=repr(error.__cause__) if error.__cause__ else None,elapsed_seconds=time.time()-clock['started_unix'])
            raise

def _exists(session):return subprocess.run(['tmux','has-session','-t','='+session],capture_output=True).returncode==0

def launch_study(root,config):
    root=Path(root).resolve();artifact=root/'artifacts/notebook_08'/config['run_id']
    with run_writer(artifact/'launch_lock'):
        artifact,binding,clock=_prepare(root,config);session='fmm-replication-'+config['run_id']
        if _exists(session):
            _load(root,'deadline_watchdog').launch_watchdog(root,config['run_id'])
            return {'status':'EXISTING_SESSION','session':session,'deadline_unix':clock['deadline_unix'],'artifact':str(artifact)}
        prior=_read(artifact/'study_status.json') if (artifact/'study_status.json').exists() else {}
        if prior.get('status')=='COMPLETED':return prior
        if time.time()>=clock['deadline_unix']:raise TimeoutError('Original deadline expired; Run All cannot reset it')
        setup=next(c.source for c in nbformat.read(root/MAIN_NOTEBOOK,as_version=4).cells if c.get('id')=='bootstrap-energy-replication')
        code=setup+'\nWORKFLOW.run_study(ROOT, '+repr(config['run_id'])+')\n'
        worker=artifact/'worker.ipynb'
        if worker.exists():
            if [c.source for c in nbformat.read(worker,as_version=4).cells if c.cell_type=='code']!=[code]:raise ValueError('Detached worker source changed')
        else:
            nbformat.write(nbformat.v4.new_notebook(cells=[nbformat.v4.new_markdown_cell('# Notebook08 detached worker\nOutputs: this run folder; full-state checkpoints: checkpoints/'+config['run_id']+'/. Resume with unchanged Notebook08 Run All before the original eight-hour deadline.'),nbformat.v4.new_code_cell(code)],metadata={'kernelspec':{'display_name':'Python (ahr-go)','language':'python','name':'ahr-go'}}),worker)
        command=[sys.executable,'-m','jupyter','nbconvert','--execute','--to','notebook','--inplace','--ExecutePreprocessor.timeout=-1','--ExecutePreprocessor.kernel_name=ahr-go',str(worker)]
        shell='exec '+shlex.join(command)+' > '+shlex.quote(str(artifact/'worker.log'))+' 2>&1'
        launch={'session':session,'artifact':str(artifact),'command':command,'attach':'tmux attach -t '+session,'detach':'Ctrl-b, then d','deadline_unix':clock['deadline_unix'],'deadline_utc':datetime.fromtimestamp(clock['deadline_unix'],timezone.utc).isoformat(),'binding_hash':_hash(binding)}
        atomic_json(artifact/'launch.json',launch);_status(artifact,'launching','STARTING')
        try:
            _load(root,'deadline_watchdog').launch_watchdog(root,config['run_id'])
            subprocess.run(['tmux','new-session','-d','-s',session,'-c',str(root),'bash','-c',shell],check=True)
        except BaseException as error:_status(artifact,'launch_failed','FAILED',error=repr(error));raise
        return {'status':'LAUNCHED',**launch}

def inspect_study(root,run_id):
    artifact=Path(root)/'artifacts/notebook_08'/run_id
    if not (artifact/'study_status.json').exists():return {'status':'NOT_LAUNCHED'}
    status=_read(artifact/'study_status.json');clock=_read(artifact/'budget_clock.json')
    result={**status,'session':'fmm-replication-'+run_id,'deadline_utc':datetime.fromtimestamp(clock['deadline_unix'],timezone.utc).isoformat(),'remaining_minutes':max(0,(clock['deadline_unix']-time.time())/60)}
    rows=[]
    for p in sorted((Path(root)/'checkpoints'/run_id/'stages/learning').glob('arms/*/manifest.json')):
        m=_read(p);rows.append({'arm':p.parent.name,'successful_updates':m.get('successful_updates',m.get('latest_step'))})
    result['saved_training_progress']=rows
    return result
